# Fine-tuning

Run with a **T4 GPU** runtime. Section 1 runs every session. Section 2 (checks) runs once, before the first long run. Section 3 starts or resumes a run; rerun it after a disconnect.

In [ ]:
REPO_URL = "https://github.com/davidyi0/handwritten-math-to-latex.git"
REPO_DIR = "/content/CoMER"
DRIVE_DATA = "/content/drive/MyDrive/hme100k"
LOCAL_DATA = "/content/hme100k"
RUNS = "/content/drive/MyDrive/comer_runs"
CONFIG = "ours/configs/baseline.yaml"
RUN = "baseline"

## 1. Every session

In [ ]:
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

from ours.colab_setup import setup
setup()

In [ ]:
!python -m ours.fetch_data --local {LOCAL_DATA} --drive {DRIVE_DATA}
!python -m ours.expand_vocab

## 2. Checks (once)

Zero-shot score in 32-bit and 16-bit on 200 validation images: the scores should be close, and the 32-bit one is the "before" number.

In [ ]:
!python -m ours.finetune validate -c {CONFIG} --ckpt_path null --run_dir /content/zs32 --trainer.precision 32 --data.val_limit 200
!python -m ours.finetune validate -c {CONFIG} --ckpt_path null --run_dir /content/zs16 --trainer.precision 16 --data.val_limit 200

Overfit 2 batches: train loss should approach 0. val_ExpRate stays low, since Lightning scores it on validation images the model never trains on.

In [ ]:
!python -m ours.finetune fit -c {CONFIG} --run_dir /content/overfit --trainer.overfit_batches 2 --trainer.max_epochs 60 --trainer.check_val_every_n_epoch 20

Timing: 300 training batches, with validation cut to 1 batch (the LR scheduler needs val_ExpRate). Seconds per batch (from the progress bar) times the train batch count (printed at the start) gives the epoch time.

In [ ]:
!python -m ours.finetune fit -c {CONFIG} --run_dir /content/timing --trainer.max_epochs 1 --trainer.limit_train_batches 300 --trainer.limit_val_batches 1

## 3. Train (rerun to resume)

In [ ]:
!mkdir -p {RUNS}/{RUN}
!python -m ours.finetune fit -c {CONFIG} --run_dir {RUNS}/{RUN} 2>&1 | tee -a {RUNS}/{RUN}/train_log.txt

from google.colab import runtime
runtime.unassign()